# 05. Design hyetographs

This notebook constructs 24-h design storms with 1-h blocks by the alternating block method (Chow et al. 1988) for the 2-, 5-, 10-, 25-, 50- and 100-year return periods under the observed baseline and the four future cases. It produces the design hyetographs and the table of peak intensities. The 2-, 10-, 50- and 100-year hyetographs were applied in HEC-RAS as spatially uniform rain-on-grid precipitation.

## Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path("..")
CASES = {   # climate case -> (label, design-rainfall table)
    "observed":         ("Observed",             ROOT / "results" / "02_frequency" / "design_rainfall_observed.csv"),
    "ssp245_2021_2050": ("SSP2-4.5 (2021-2050)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp245_2021_2050.csv"),
    "ssp245_2051_2080": ("SSP2-4.5 (2051-2080)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp245_2051_2080.csv"),
    "ssp585_2021_2050": ("SSP5-8.5 (2021-2050)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp585_2021_2050.csv"),
    "ssp585_2051_2080": ("SSP5-8.5 (2051-2080)", ROOT / "results" / "03_cmip6" / "design_rainfall_ssp585_2051_2080.csv"),
}
CHOSEN_DIST = "GEV"            # distribution selected in notebook 02
EXPONENT = 1.0 / 3.0           # exponent of the one-third rule
STORM_HOURS = 24               # storm duration (h)
BLOCK_MIN = 60                 # block length (min)
RETURN_PERIODS = [2, 5, 10, 25, 50, 100]
OUTDIR = ROOT / "results" / "05_hyetographs"
OUTDIR.mkdir(parents=True, exist_ok=True)

## 1. Alternating block method

The cumulative depth for a duration of $j$ blocks follows from the one-third rule, and the incremental depth of block $j$ is

$$\Delta P_j = P_{24}\left[\left(\frac{j\,\Delta t}{24}\right)^{1/3} - \left(\frac{(j-1)\,\Delta t}{24}\right)^{1/3}\right], \quad j = 1, \ldots, 24$$

with $\Delta t = 1$ h. The largest increment is placed in the 13th hour (12 to 13 h), and the remaining increments, in descending order, are placed alternately after and before it. The depth of the most intense window of every duration therefore equals the IDF depth for that duration, and the storm total equals $P_{24}$.

In [ ]:
def alternating_block(p24, T, storm_hours, block_min, exponent):
    """Hyetograph of one return period by the alternating block method."""
    dt = block_min / 60.0
    nblocks = int(round(storm_hours / dt))

    # cumulative depth for durations of 1, 2, ... blocks
    cum = [p24 * ((i * dt) / 24.0) ** exponent for i in range(1, nblocks + 1)]
    # incremental depth of each block
    inc = [cum[0]] + [cum[i] - cum[i - 1] for i in range(1, nblocks)]
    # increments in descending order
    order = np.argsort(inc)[::-1]
    # largest in the centre, the rest alternately after and before it
    seq = [None] * nblocks
    centre = nblocks // 2
    seq[centre] = inc[order[0]]
    left, right = centre - 1, centre + 1
    go_right = True
    for idx in order[1:]:
        if go_right and right < nblocks:
            seq[right] = inc[idx]; right += 1
        elif left >= 0:
            seq[left] = inc[idx]; left -= 1
        elif right < nblocks:
            seq[right] = inc[idx]; right += 1
        go_right = not go_right
    seq = [0.0 if v is None else v for v in seq]

    times = [round(i * block_min, 3) for i in range(nblocks)]
    return pd.DataFrame({
        "Time_minutes": times,
        "Time_hours": [round(t / 60.0, 4) for t in times],
        "Depth_mm": [round(v, 3) for v in seq],
        "Intensity_mm_per_hr": [round(v / dt, 3) for v in seq],
        "Cumulative_mm": [round(v, 3) for v in np.cumsum(seq)],
    })

## 2. Hyetographs for each climate case

For every hyetograph the storm total is checked against the 24-h design depth.

In [ ]:
col = f"{CHOSEN_DIST} (mm)"
peaks = {}

for case, (label, path) in CASES.items():
    design = pd.read_csv(path)
    P24 = dict(zip(design["Return period (yr)"], design[col]))
    outdir = OUTDIR / case
    outdir.mkdir(parents=True, exist_ok=True)

    summary, hyetos = [], {}
    for T in RETURN_PERIODS:
        h = alternating_block(P24[T], T, STORM_HOURS, BLOCK_MIN, EXPONENT)
        h.to_csv(outdir / f"hyetograph_{T}yr.csv", index=False)
        hyetos[T] = h
        total = h["Depth_mm"].sum()
        peak = h["Intensity_mm_per_hr"].max()
        summary.append({
            "Return period (yr)": T,
            "Storm duration (h)": STORM_HOURS,
            "Time step (min)": BLOCK_MIN,
            "Total depth (mm)": round(total, 1),
            "Design depth (mm)": round(P24[T], 1),
            "Total equals design depth": bool(abs(total - P24[T]) < 1),
            "Peak intensity (mm/hr)": round(peak, 1),
            "Peak block share (%)": round(100 * h["Depth_mm"].max() / total, 1),
            "Three largest blocks share (%)": round(100 * h["Depth_mm"].nlargest(3).sum() / total, 1),
        })
    summ = pd.DataFrame(summary)
    summ.to_csv(outdir / "hyetograph_summary.csv", index=False)
    peaks[label] = summ.set_index("Return period (yr)")["Peak intensity (mm/hr)"]

    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
    for ax, T in zip(axes.ravel(), RETURN_PERIODS):
        h = hyetos[T]
        ax.bar(h["Time_hours"], h["Intensity_mm_per_hr"],
               width=BLOCK_MIN / 60.0 * 0.9, color="#2E5C8A")
        ax.set_title(f"{T}-year  ({h['Depth_mm'].sum():.0f} mm)", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
    for ax in axes[1]:
        ax.set_xlabel("Time (hours)")
    for ax in axes[:, 0]:
        ax.set_ylabel("Intensity (mm/hr)")
    fig.suptitle(f"Design Storm Hyetographs (Alternating Block Method): {label}", fontsize=12)
    fig.tight_layout()
    fig.savefig(outdir / "fig_hyetographs.png", dpi=300)
    plt.show()

## 3. Peak hourly intensity by return period and climate case

In [ ]:
peak_table = pd.DataFrame(peaks)
peak_table.to_csv(OUTDIR / "peak_intensity.csv")
peak_table